In [1]:
# NE PAS faire "!pip install torch" ici : dans le noyau GPU, pip pourrait remplacer
# la version ROCm par la version CPU de PyPI et casser l'accélération.
# On se contente de VÉRIFIER ce qui est installé.
import torch

print(f"torch        : {torch.__version__}")
print(f"GPU utilisable : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU          : {torch.cuda.get_device_name(0)}")
else:
    print("→ Tu es sur le noyau CPU. Kernel → Change Kernel → « Python (GPU ROCm) »")

torch        : 2.9.1+rocm7.2.1
GPU utilisable : True
GPU          : AMD Radeon(TM) 880M Graphics


In [2]:
# lightning et torchmetrics sont déjà installés dans le noyau GPU. Simple vérification.
import pytorch_lightning as pl
import torchmetrics

print(f"pytorch_lightning : {pl.__version__}")
print(f"torchmetrics      : {torchmetrics.__version__}")

pytorch_lightning : 2.6.5
torchmetrics      : 1.9.0


In [3]:
import numpy as np 
import torch
from torch import nn
from torch.utils.data import Dataset
import pytorch_lightning as pl
from torchmetrics import MeanSquaredError
import matplotlib.pyplot as plt

pl.seed_everything(32)

Seed set to 32


32

In [4]:
scalar = torch.tensor(3.0)
vector = torch.tensor([3.0,1.0])
matrix = torch.tensor([[3.0, 1.0], [3.0, 1.0]])
print(f" scalar {scalar} shape {scalar.shape}")
print(f" vector {vector} shape {vector.shape}")
print(f" matrix {matrix} shape {matrix.shape}")

 scalar 3.0 shape torch.Size([])
 vector tensor([3., 1.]) shape torch.Size([2])
 matrix tensor([[3., 1.],
        [3., 1.]]) shape torch.Size([2, 2])


In [5]:
a = np.array([1.0, 2.0, 3.0])
b = np.array([[1.0, 2.0, 3.0]])

In [6]:
a * b 

array([[1., 4., 9.]])

In [7]:
np_array = np.array([1.0, 2.0, 3.0])
torch_tensor = torch.tensor([1.0, 2.0, 3.0])

In [8]:
print(f" Numpy result {np_array * 2}")
print(f" Torch result {torch_tensor * 2}")

 Numpy result [2. 4. 6.]
 Torch result tensor([2., 4., 6.])


In [9]:
print(f" Numpy result {np_array * np_array}")
print(f" Torch result {torch_tensor * torch_tensor}")

 Numpy result [1. 4. 9.]
 Torch result tensor([1., 4., 9.])


In [10]:
print(f" Numpy result {np_array + np_array}")
print(f" Torch result {torch_tensor + torch_tensor}")

 Numpy result [2. 4. 6.]
 Torch result tensor([2., 4., 6.])


In [11]:
print(f" Numpy result {np_array.mean()}")
print(f" Torch result {torch_tensor.mean()}")

 Numpy result 2.0
 Torch result 2.0


In [12]:
# Sur AMD, l'API garde le nom "cuda" : c'est voulu, ton code reste identique à un tuto NVIDIA.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Selected device {device}")

if device.type == "cuda":
    print(f"→ {torch.cuda.get_device_name(0)} (via ROCm {torch.version.hip.split('-')[0]})")
else:
    print("→ CPU. Pour le GPU : Kernel → Change Kernel → « Python (GPU ROCm) »")

Selected device cuda
→ AMD Radeon(TM) 880M Graphics (via ROCm 7.2.53211)


In [13]:
x_cpu = torch.tensor([1.0, 2.0, 3.0])

In [14]:
device

device(type='cuda')

In [15]:
x_device = x_cpu.to(device)

In [16]:
x_device 

tensor([1., 2., 3.], device='cuda:0')

In [17]:
x = torch.tensor(2.0, requires_grad=True)

In [18]:
y = x ** 2 + 3 * x + 1
y.backward()

In [19]:
y 

tensor(11., grad_fn=<AddBackward0>)

In [20]:
dy = 2*x + 3
dy 

tensor(7., grad_fn=<AddBackward0>)

In [21]:
print(f"x: {x} y: {y}")
print(f"dy/dx at x = 2 {x.grad.item()}")

x: 2.0 y: 11.0
dy/dx at x = 2 7.0


In [22]:
x.grad

tensor(7.)

In [23]:
x = torch.tensor(4.0, requires_grad=True)
y = x**4 + 4
y.backward()
print(f"dy/dx at x = 4.0 {x.grad.item()}")

dy/dx at x = 4.0 256.0


## Neural Networks


In [24]:
class TinyNetwork(nn.Module):
    # CORRIGÉ : in_features est maintenant un paramètre.
    # Avant, une 2e classe du même nom (plus bas) écrasait celle-ci et cassait tout.
    # CORRIGÉ : "selt" -> "self", "actication" -> "activation"
    def __init__(self, in_features=1, hidden=4):
        super().__init__()
        self.layer1 = nn.Linear(in_features=in_features, out_features=hidden)
        self.activation = nn.ReLU()
        self.layer2 = nn.Linear(in_features=hidden, out_features=1)

    def forward(self, x):
        x = self.layer1(x)
        x = self.activation(x)
        x = self.layer2(x)
        return x

In [25]:
model = TinyNetwork().to(device)                  # 1 entrée. .to(device) -> envoie le modèle sur le GPU
example_input = torch.tensor([[2.0]]).to(device)  # les données doivent être sur le MÊME device que le modèle
example_output = model(example_input)
print(example_output)
print(f"nb de paramètres : {sum(p.numel() for p in model.parameters())}")

tensor([[0.4460]], device='cuda:0', grad_fn=<AddmmBackward0>)
nb de paramètres : 13


In [26]:
# Variante 2 entrées / couche cachée plus large.
# AVANT : c'était une 2e "class TinyNetwork" qui ÉCRASAIT la première -> l'entraînement
# plus bas plantait, car tes données n'ont qu'1 colonne. Maintenant c'est un paramètre.
model_2_entrees = TinyNetwork(in_features=2, hidden=8).to(device)

exemple = torch.tensor([[2.0, 5.0]]).to(device)   # 2 colonnes cette fois
print(model_2_entrees(exemple))
print(f"nb de paramètres : {sum(p.numel() for p in model_2_entrees.parameters())}")

tensor([[-0.0878]], device='cuda:0', grad_fn=<AddmmBackward0>)
nb de paramètres : 33


In [27]:
x = torch.tensor([[0.0],[1.0],[2.0],[3.0]])
y = torch.tensor([[1.0],[2.0],[4.0],[6.0]])

In [28]:
from torch.utils.data import Dataset, DataLoader


class TinyDataset(Dataset):
    def __init__(self):
        self.x = torch.tensor([[0.0], [1.0], [2.0], [3.0]])
        self.y = torch.tensor([[1.0], [2.0], [4.0], [6.0]])

    def __len__(self):
        return len(self.x)

    def __getitem__(self, index):
        # CORRIGÉ : renvoyait (x, y, 1). Ce 3e élément ne servait à rien et provoquait
        # "ValueError: too many values to unpack" dans la boucle d'entraînement.
        # Un Dataset standard renvoie (entrée, cible).
        return self.x[index], self.y[index]


dataset = TinyDataset()
print(len(dataset))

x, y = dataset[3]
print(f"x:{x} - y:{y}")

loader = DataLoader(dataset, batch_size=2, shuffle=False)

for batch_x, batch_y in loader:
    print(f"batch_x = {batch_x}")

4
x:tensor([3.]) - y:tensor([6.])
batch_x = tensor([[0.],
        [1.]])
batch_x = tensor([[2.],
        [3.]])


In [31]:
pure_model = TinyNetwork().to(device)   # le modèle part sur le GPU
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(pure_model.parameters(), lr=0.01)

# CORRIGÉ : 3 epochs n'apprenaient presque rien -> 200 pour voir la perte descendre.
# Le DataLoader sort des tenseurs CPU : il faut les envoyer sur le GPU à chaque batch.
for epoch in range(200):

    model.train()
    total_train_loss = 0
    average_total_error =0 
    for i, (batch_x, batch_y) in enumerate(loader):
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        prediction = pure_model(batch_x)
        print(f"inputs : {batch_x} prediction : {prediction}")
        loss = loss_fn(prediction, batch_y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item()*batch_x.size(0)
    average_total_error += total_train_loss/len(loader.dataset)
    if epoch % 40 == 0 or epoch == 199:
        print(f"epoch {epoch:>3} - loss = {loss.item():.4f}")

print(f"\nLe modèle tourne sur : {next(pure_model.parameters()).device}")

inputs : tensor([[0.],
        [1.]], device='cuda:0') prediction : tensor([[-0.2095],
        [-0.2095]], device='cuda:0', grad_fn=<AddmmBackward0>)
inputs : tensor([[2.],
        [3.]], device='cuda:0') prediction : tensor([[-0.1753],
        [-0.1753]], device='cuda:0', grad_fn=<AddmmBackward0>)
epoch   0 - loss = 27.7834
inputs : tensor([[0.],
        [1.]], device='cuda:0') prediction : tensor([[-0.0718],
        [-0.0718]], device='cuda:0', grad_fn=<AddmmBackward0>)
inputs : tensor([[2.],
        [3.]], device='cuda:0') prediction : tensor([[-0.0403],
        [-0.0403]], device='cuda:0', grad_fn=<AddmmBackward0>)
inputs : tensor([[0.],
        [1.]], device='cuda:0') prediction : tensor([[0.0605],
        [0.0605]], device='cuda:0', grad_fn=<AddmmBackward0>)
inputs : tensor([[2.],
        [3.]], device='cuda:0') prediction : tensor([[0.0893],
        [0.0893]], device='cuda:0', grad_fn=<AddmmBackward0>)
inputs : tensor([[0.],
        [1.]], device='cuda:0') prediction : tensor([[